# Can we predict `late_delivery`?
Plan follows checklist.md. Prediction is for future consignments, so the split is temporal. Each consignment is one row, so no entity grouping is needed (checked below).

**Feature policy (checklist 7):** `consignment` is an identifier, dropped. `claim_paid_eur` is compensation paid on a claim settled *after* delivery, so it is only known once lateness is known (it's ~0 for on-time rows). It is target leakage and is dropped. Features: `carrier`, `distance_km`, `weight_kg`, `parcels`, and hour/weekday of `dispatched_on`, all known at dispatch.

In [ ]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
assert df.consignment.is_unique
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
print(df.shape)

## 1. Hold out the test set (latest 20% of rows, by time) before any exploration of outcomes

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.min(),train.dispatched_on.max()); print(test.dispatched_on.min(),test.dispatched_on.max())
FEATS=['carrier','distance_km','weight_kg','parcels','hour','dow']
Xtr,ytr=train[FEATS],train.late_delivery; Xte,yte=test[FEATS],test.late_delivery

## 2. Exploration on training rows only
The late rate varies a lot over time, so the base rate is not stable.

In [ ]:
print(train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.mean().round(3).to_string())
print(train.groupby('carrier').late_delivery.mean())

## 3. Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial)
2. `logreg_C{0.1,1,10}`: one-hot carrier + scaled numerics, L2 logistic regression
3. `hgb_d{2,4}`: histogram gradient boosting, learning_rate 0.05, 200 iters

Selection metric: mean CV average precision. The decision threshold is then chosen on out-of-fold predictions to maximise F1. CV is expanding-window forward-chaining inside the training rows. Preprocessing sits inside each pipeline, so it is refit per fold.

In [ ]:
def pre():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['carrier']),
                              ('n',StandardScaler(),['distance_km','weight_kg','parcels','hour','dow'])])
cands={'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))])}
for C in [0.1,1,10]: cands[f'logreg_C{C}']=Pipeline([('p',pre()),('m',LogisticRegression(C=C,max_iter=1000))])
for d in [2,4]: cands[f'hgb_d{d}']=Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=0))])
tscv=TimeSeriesSplit(n_splits=5)
res={};oof={}
for name,est in cands.items():
    aps=[];aucs=[];p_all=np.full(len(Xtr),np.nan)
    for a,b in tscv.split(Xtr):
        e=clone(est).fit(Xtr.iloc[a],ytr.iloc[a]); p=e.predict_proba(Xtr.iloc[b])[:,1]; p_all[b]=p
        aps.append(average_precision_score(ytr.iloc[b],p)); aucs.append(roc_auc_score(ytr.iloc[b],p))
    res[name]=(np.mean(aps),np.std(aps),np.mean(aucs)); oof[name]=p_all
print(pd.DataFrame(res,index=['cv_AP','sd_AP','cv_AUC']).T.round(3))

## 4. Choose model and threshold on CV only

In [ ]:
best=max((n for n in res if n!='baseline'),key=lambda n:res[n][0])
mask=~np.isnan(oof[best]); ths=np.linspace(0.05,0.9,86)
f1s=[f1_score(ytr[mask],oof[best][mask]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| CV AP',round(res[best][0],3),'| baseline CV AP',round(res['baseline'][0],3),'| threshold',round(thr,2),'| OOF F1',round(max(f1s),3))

**Decision recorded before the test set is scored:** model = `best` printed above, features = `carrier, distance_km, weight_kg, parcels, hour, dow`, threshold = value printed above. Nothing is changed after the test score.

## 5. Refit on all training rows; score the test set once

In [ ]:
final=clone(cands[best]).fit(Xtr,ytr); p=final.predict_proba(Xte)[:,1]
base=clone(cands['baseline']).fit(Xtr,ytr).predict_proba(Xte)[:,1]
print('test late rate',round(yte.mean(),4),'| n',len(yte))
print('AUC',round(roc_auc_score(yte,p),3),'| AP',round(average_precision_score(yte,p),3),'(no-skill AP = prevalence',round(yte.mean(),3),')')
print('Brier',round(brier_score_loss(yte,p),4),'vs baseline',round(brier_score_loss(yte,base),4))
print('F1 @ chosen threshold',round(f1_score(yte,p>=thr),3),'| predicted-positive rate',round((p>=thr).mean(),3))